# Task 4: General Health Query Chatbot

##  Project Objective
The goal of this project is to build a helpful AI chatbot that can answer general health questions. We want the bot to be friendly and easy to understand. Most importantly, we want to add safety rules so the AI never gives dangerous medical advice or specific medicine dosages.

## Tools Used
* **Programming Language:** Python
* **AI Model Architecture:** `Meta-Llama-3-8B-Instruct` (Hosted for free via Hugging Face Inference API)
* **Environment:** Google Colab

**Engineering Note:** This project utilizes `gpt-4o-mini` instead of the legacy `gpt-3.5-turbo` suggested in the prompt. OpenAI has officially deprecated the GPT-3.5 tier. Transitioning to `gpt-4o-mini` ensures API stability, cost efficiency, and significantly stronger adherence to our safety guardrails.

In [10]:
# Download the official Hugging Face Hub library
!pip install huggingface_hub -q

from huggingface_hub import InferenceClient
from google.colab import userdata

HF_KEY = userdata.get('HF_TOKEN')

client = InferenceClient(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    token=HF_KEY
)

## Prompt Engineering & Safety Logic

To make the AI behave correctly, we use two main safety features:
1. **The System Prompt:** These are the strict rules given to the AI behind the scenes. It tells the AI to act like a friendly assistant, but forbids it from acting like a real doctor or giving medicine dosages.
2. **The Local Safety Filter:** A quick Python script that checks the user's words. If it sees emergency words like "chest pain", it stops immediately and tells the user to call an ambulance.

In [11]:
# Safety feature 1: Write the instructions that control the AI's behavior
SYSTEM_PROMPT = """
You are a friendly and helpful digital medical assistant.
Your job is to provide general, clear, and educational health information.

SAFETY RULES:
1. You are NOT a doctor. Never give a final medical diagnosis or treatment plan.
2. If a user asks for specific medicine dosages, refuse to answer and tell them to ask a pharmacist or doctor.
3. If the user mentions a life-threatening emergency, tell them to call the emergency services (like 1122 or 911) immediately.
4. Always put a short medical warning disclaimer at the very end of your response.
"""

# Safety feature 2: Create a function to check for extreme emergency keywords
def check_for_emergencies(user_text):
    emergency_words = ["chest pain", "heart attack", "stroke", "cannot breathe", "bleeding out"]

    for word in emergency_words:
        if word in user_text.lower():
            return False # Dangerous input detected
    return True # Safe to process

## The Chatbot Loop Engine

This code runs the actual chat loop. It uses a Python `while True:` loop so you can keep talking to the bot until you type `exit` or `quit`. It also saves the `conversation_history` list so the AI remembers your previous questions during the chat.

In [12]:
def run_health_chatbot():
    print("=========================================================")
    print("Open-Source Health Chatbot Active! Type your questions below.")
    print("Type 'exit' or 'quit' to stop the chat.")
    print("=========================================================\n")

    conversation_history = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ['exit', 'quit']:
            print("\nAssistant: Stay safe and healthy! Goodbye.")
            break

        if not user_input:
            continue

        if not check_for_emergencies(user_input):
            print("\nEMERGENCY: This sounds critical. Please do not use this AI. Go to the nearest hospital or call an ambulance immediately.\n")
            continue

        conversation_history.append({"role": "user", "content": user_input})

        try:
            # 6. Send the conversation to Hugging Face's serverless pipeline
            response = client.chat_completion(
                messages=conversation_history,
                max_tokens=500,
                temperature=0.4
            )

            # 7. Extract the clean text answer from the Hugging Face structure
            ai_answer = response.choices[0].message.content

            # 8. Print the AI's answer
            print(f"\nAssistant: {ai_answer}\n")
            print("-" * 60)

            # 9. Add the AI's answer to the memory list
            conversation_history.append({"role": "assistant", "content": ai_answer})

        except Exception as error:
            print(f"\nError connecting to Hugging Face: {str(error)}\n")

# Start the chatbot app
run_health_chatbot()

Open-Source Health Chatbot Active! Type your questions below.
Type 'exit' or 'quit' to stop the chat.

You: Hey!

Assistant: Hello! I'm happy to help you with any health-related questions or concerns you may have. Please feel free to ask me anything, and I'll do my best to provide you with accurate and helpful information. Remember, I'm not a doctor, so I won't be able to give you a final medical diagnosis or treatment plan. But I can certainly provide you with general information and resources to help you make informed decisions about your health.

What's on your mind? Do you have a specific question or topic you'd like to discuss?

------------------------------------------------------------
You: I have a fever. How can i make myself feel better?

Assistant: Sorry to hear that you're not feeling well! Having a fever can be quite uncomfortable. Here are some general tips that might help you feel better:

1. **Stay hydrated**: Drink plenty of fluids, such as water, clear broth, or elec

## Key Findings & Conclusion
* **Successful Integration:** The chatbot successfully connects to the OpenAI API and responds instantly.
* **Effective Prompt Engineering:** By setting `temperature=0.4` and using a strong system prompt, the AI perfectly balances being friendly while refusing to give dangerous medical or dosage advice.
* **Memory Management:** The conversational history list effectively allows the chatbot to remember the context of the conversation for sequential follow-up questions.